In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pygwalker as pyg
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
#Importing feature_extraction.csv file

feat_df = pd.read_csv("feature_extraction.csv")
#feat_df["max_complexity"].max()
feat_df.head()

,file name,loc,sloc,comment_ratio,blank_ratio,cyclomatic_complexity,max_complexity,maintainability_index,halstead_effort,num_functions,...,commit_count,author_count,days_since_change,bug_fix_commits,churn_rate,refactor_commits,dependency_count,max_nesting_depth,is_tested,label
0,wicked-garden/scripts\reset.py,222,169,0.085586,0.135135,9.666667,58,46.077070,1517.398123,6,...,2,1,33,1,368,1,6,6,0,0
1,wicked-garden/scripts\resolve_path.py,22,8,0.363636,0.272727,0.000000,0,98.817997,2.377444,0,...,2,1,34,2,24,0,3,1,0,1
2,wicked-garden/scripts\_agents.py,357,179,0.285714,0.198880,4.733333,71,50.826472,7470.287739,15,...,3,1,25,2,601,2,5,6,0,1
3,wicked-garden/scripts\_capability_registry.py,226,196,0.106195,0.061947,1.000000,2,100.000000,0.000000,2,...,1,1,25,1,226,1,2,1,0,0
4,wicked-garden/scripts\_capability_resolver.py,322,196,0.217391,0.180124,14.400000,72,49.128629,4896.134033,5,...,1,1,25,1,322,1,9,8,0,0


I have anormal values here, since i will be using random forest (explain details later); these will interfere with the model training (make better notes for the complete version of this notebook)

In [3]:
type(feat_df["label"][0])

numpy.int64

In [4]:
feat_df.shape

(2503, 21)

In [20]:
#Feat separation from label 
X = feat_df.iloc[:, 1:20]
Y = feat_df.iloc[:, -1:]

      label
0         0
1         1
2         1
3         0
4         0
...     ...
2498      1
2499      1
2500      1
2501      1
2502      1

[2503 rows x 1 columns]


Data manipulation

In [6]:

limit = feat_df["churn_rate"].quantile(0.99)
feat_df["churn_rate"] = feat_df["churn_rate"].clip(upper=limit)
print(feat_df["churn_rate"].max())

4008.6000000000004


In [7]:
feat_df["churn_rate"].head()

0    368.0
1     24.0
2    601.0
3    226.0
4    322.0
Name: churn_rate, dtype: float64

In [24]:
#splitting

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [9]:
#revisar muestras 

# 1. Crear el DataFrame directamente con las características
results = pd.DataFrame(X_train)

# 2. Agregar la columna de etiquetas (variable objetivo)
results['labels'] = y_train

# 3. Lanzar PyGWalker
walker = pyg.walk(results)

Box(children=(HTML(value='\n<div id="ifr-pyg-0006562245671787K91ObWlsDqk8wXgn" style="height: auto">\n    <hea…

In [38]:
random_forest = RandomForestClassifier()
random_forest.fit(X_train, y_train)

y_pred = random_forest.predict(X_test)
random_forest.score(X_test, y_test)

1.0

intento 1: 100% de score. Esto claramente no esta bien. Estamos teniendo un error de diseño que esta causando una posible fuga de datos. 

In [35]:
# Excluir los features contaminados por el mismo texto de commit que define el label
features_sin_leakage = [c for c in X_train.columns if c not in ['bug_fix_commits', 'refactor_commits']]

X_train_v2 = X_train[features_sin_leakage]
X_test_v2 = X_test[features_sin_leakage]

modelo_v2 = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_v2.fit(X_train_v2, y_train)

y_pred_v2 = modelo_v2.predict(X_test_v2)

print("Score:", modelo_v2.score(X_test_v2, y_test))
print()
print(classification_report(y_test, y_pred_v2))
print()
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_v2))
print()

importancias_v2 = pd.Series(modelo_v2.feature_importances_, index=X_train_v2.columns)
print(importancias_v2.sort_values(ascending=False))

Score: 0.8822355289421158

              precision    recall  f1-score   support

           0       0.88      0.94      0.91       311
           1       0.88      0.79      0.84       190

    accuracy                           0.88       501
   macro avg       0.88      0.87      0.87       501
weighted avg       0.88      0.88      0.88       501


Matriz de confusión:
[[291  20]
 [ 39 151]]

commit_count             0.291418
author_count             0.116588
churn_rate               0.089825
days_since_change        0.069367
sloc                     0.056849
dependency_count         0.056733
loc                      0.052908
maintainability_index    0.036775
avg_function_len         0.036571
halstead_effort          0.030689
max_complexity           0.029564
cyclomatic_complexity    0.029148
blank_ratio              0.028619
comment_ratio            0.027570
num_functions            0.023555
max_nesting_depth        0.017170
is_tested                0.006651
dtype: float64


el modelo actual usa features de todo el historial del repo; una versión futura con corte temporal permitiría validar predicción genuina hacia adelante.